## Loading and understanding the data

In [1]:
import pandas as pd
import numpy as np
import time

df = pd.read_csv("garments_worker_productivity.csv")

df.head()

,date,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,1/1/2015,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,0.940725
1,1/1/2015,Quarter1,finishing,Thursday,1,0.75,3.94,NaN,960,0,0.0,0,0,8.0,0.886500
2,1/1/2015,Quarter1,sweing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
3,1/1/2015,Quarter1,sweing,Thursday,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
4,1/1/2015,Quarter1,sweing,Thursday,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0,0.800382


In [2]:
print("Number of rows: ", df.shape[0])
print("Number of columns: ", df.shape[1])



Number of rows:  1197
Number of columns:  15


In [3]:
df.isnull().sum()

date                       0
quarter                    0
department                 0
day                        0
team                       0
targeted_productivity      0
smv                        0
wip                      506
over_time                  0
incentive                  0
idle_time                  0
idle_men                   0
no_of_style_change         0
no_of_workers              0
actual_productivity        0
dtype: int64

In [5]:
categorical_columns = ['quarter', 'department', 'day']

for column in categorical_columns:
    print("\n", column)
    print(df[column].value_counts())


 quarter
quarter
Quarter1    360
Quarter2    335
Quarter4    248
Quarter3    210
Quarter5     44
Name: count, dtype: int64

 department
department
sweing        691
finishing     257
finishing     249
Name: count, dtype: int64

 day
day
Wednesday    208
Sunday       203
Tuesday      201
Thursday     199
Monday       199
Saturday     187
Name: count, dtype: int64


In [10]:
df['quarter'] = df['quarter'].str.strip()
df['department'] = df['department'].str.strip()
df['day'] = df['day'].str.strip()

df['MeetsTarget'] = (
    df['actual_productivity'] >= df['targeted_productivity']
).astype(int)

print(df['MeetsTarget'].value_counts())
print("\nPercentage")
print(df['MeetsTarget'].value_counts(normalize=True) * 100)

y_reg = df['actual_productivity']
y_clf = df['MeetsTarget']

MeetsTarget
1    875
0    322
Name: count, dtype: int64

Percentage
MeetsTarget
1    73.099415
0    26.900585
Name: proportion, dtype: float64


## Part A - Scikit-learn Implementation

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LinearRegression, LogisticRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

X_reg = df.drop(columns=['actual_productivity', 'MeetsTarget', 'date'])
y_reg = df['actual_productivity']

print("X shape: ", X_reg.shape)
print("y shape: ",y_reg.shape)

X shape:  (1197, 13)
y shape:  (1197,)


### Identifying numerical and categorical features

In [14]:
categorical_features = ['quarter', 'department', 'day']
numerical_features = [
    col for col in X_reg.columns
    if col not in categorical_features
]

print("Categorical features: ")
print(categorical_features)

print("\nNumerical Features: ")
print(numerical_features)




Categorical features: 
['quarter', 'department', 'day']

Numerical Features: 
['team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers']


### Train-test split, preprocessing and linear regression pipeline

In [59]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers = [
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

linear_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

start_time = time.perf_counter()
linear_model.fit(X_train_reg, y_train_reg)
end_time = time.perf_counter()
linear_training_time = end_time - start_time


start_time = time.perf_counter()
y_pred_reg = linear_model.predict(X_test_reg)
end_time = time.perf_counter()
linear_prediction_time = end_time - start_time

print(
    "Linear Regression Prediction Time: ",
    linear_prediction_time, "seconds"
)


Linear Regression Prediction Time:  0.009557700017467141 seconds


In [60]:
mae = mean_absolute_error(y_test_reg, y_pred_reg)

rmse = np.sqrt(
    mean_squared_error(y_test_reg, y_pred_reg)
)

r2 = r2_score(y_test_reg, y_pred_reg)

print("SCIKIT-LEARN LINEAR REGRESSION")
print("--------------------------------")
print(f"MAE             : {mae:.4f}")
print(f"RMSE            : {rmse:.4f}")
print(f"R2 Score        : {r2:.4f}")
print(f"Training Time   : {linear_training_time:.6f} seconds")
print(f"Prediction Time : {linear_prediction_time:.6f} seconds")

SCIKIT-LEARN LINEAR REGRESSION
--------------------------------
MAE             : 0.1088
RMSE            : 0.1481
R2 Score        : 0.1736
Training Time   : 0.169736 seconds
Prediction Time : 0.009558 seconds


In [61]:
X_clf = df.drop(
    columns=['actual_productivity', 'MeetsTarget', 'date']
)
y_clf = df['MeetsTarget']

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf,
    y_clf,
    test_size=0.2,
    random_state=42
)

logistic_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

start_time = time.perf_counter()
logistic_model.fit(X_train_clf,y_train_clf )
end_time = time.perf_counter()
logistic_training_time = end_time - start_time

start_time = time.perf_counter()
y_pred_clf = logistic_model.predict(X_test_clf)
end_time = time.perf_counter()
logistic_prediction_time = end_time - start_time

accuracy = accuracy_score(y_test_clf, y_pred_clf)
precision = precision_score(y_test_clf, y_pred_clf)
recall = recall_score(y_test_clf, y_pred_clf)
f1 = f1_score(y_test_clf, y_pred_clf)

print("SCIKIT-LEARN LOGISTIC REGRESSION")
print("----------------------------------")
print(f"Accuracy        : {accuracy:.4f}")
print(f"Precision       : {precision:.4f}")
print(f"Recall          : {recall:.4f}")
print(f"F1 Score        : {f1:.4f}")
print(f"Training Time   : {logistic_training_time:.6f} seconds")
print(f"Prediction Time : {logistic_prediction_time:.6f} seconds")

SCIKIT-LEARN LOGISTIC REGRESSION
----------------------------------
Accuracy        : 0.7500
Precision       : 0.7696
Recall          : 0.9435
F1 Score        : 0.8477
Training Time   : 0.139415 seconds
Prediction Time : 0.010438 seconds


## Part B - From-Scratch Implementation

### Linear regression from scratch

In [94]:
# ============================================================
# PART B1 - LINEAR REGRESSION FROM SCRATCH
# NumPy + Pandas only
# ============================================================

print("=" * 55)
print("FROM-SCRATCH LINEAR REGRESSION")
print("=" * 55)

# ------------------------------------------------------------
# 1. Use the SAME train-test samples from Part A
# ------------------------------------------------------------

X_train_reg_manual = X_train_reg.copy()
X_test_reg_manual = X_test_reg.copy()

y_train_reg_manual = y_train_reg.copy()
y_test_reg_manual = y_test_reg.copy()


# ------------------------------------------------------------
# 2. Handle missing values manually
# ------------------------------------------------------------

# Calculate median using TRAINING data only
wip_median_reg = X_train_reg_manual['wip'].median()

X_train_reg_manual['wip'] = (
    X_train_reg_manual['wip'].fillna(wip_median_reg)
)

X_test_reg_manual['wip'] = (
    X_test_reg_manual['wip'].fillna(wip_median_reg)
)


# ------------------------------------------------------------
# 3. One-hot encode categorical features using Pandas
# ------------------------------------------------------------

categorical_features_manual = [
    'quarter',
    'department',
    'day'
]

X_train_reg_manual = pd.get_dummies(
    X_train_reg_manual,
    columns=categorical_features_manual,
    dtype=float
)

X_test_reg_manual = pd.get_dummies(
    X_test_reg_manual,
    columns=categorical_features_manual,
    dtype=float
)

# Make test columns exactly match training columns
X_test_reg_manual = X_test_reg_manual.reindex(
    columns=X_train_reg_manual.columns,
    fill_value=0
)


# ------------------------------------------------------------
# 4. Scale numerical features manually
# ------------------------------------------------------------

numeric_features_manual = [
    'team',
    'targeted_productivity',
    'smv',
    'wip',
    'over_time',
    'incentive',
    'idle_time',
    'idle_men',
    'no_of_style_change',
    'no_of_workers'
]

# Calculate mean and standard deviation using TRAINING data only
reg_means = X_train_reg_manual[
    numeric_features_manual
].mean()

reg_stds = X_train_reg_manual[
    numeric_features_manual
].std(ddof=0)

# Avoid division by zero
reg_stds = reg_stds.replace(0, 1)

# Scale training data
X_train_reg_manual[numeric_features_manual] = (
    X_train_reg_manual[numeric_features_manual]
    - reg_means
) / reg_stds

# Scale test data using TRAINING mean and std
X_test_reg_manual[numeric_features_manual] = (
    X_test_reg_manual[numeric_features_manual]
    - reg_means
) / reg_stds


# ------------------------------------------------------------
# 5. Convert Pandas data to NumPy arrays
# ------------------------------------------------------------

X_train_reg_np = X_train_reg_manual.to_numpy(dtype=float)
X_test_reg_np = X_test_reg_manual.to_numpy(dtype=float)

y_train_reg_np = y_train_reg_manual.to_numpy(dtype=float)
y_test_reg_np = y_test_reg_manual.to_numpy(dtype=float)


# ------------------------------------------------------------
# 6. Add intercept/bias column
# ------------------------------------------------------------

X_train_reg_bias = np.c_[
    np.ones(X_train_reg_np.shape[0]),
    X_train_reg_np
]

X_test_reg_bias = np.c_[
    np.ones(X_test_reg_np.shape[0]),
    X_test_reg_np
]


# ------------------------------------------------------------
# 7. Train Linear Regression manually
# ------------------------------------------------------------

start_time = time.perf_counter()

# Closed-form solution using pseudo-inverse
beta_manual = (
    np.linalg.pinv(X_train_reg_bias)
    @ y_train_reg_np
)

end_time = time.perf_counter()

manual_linear_training_time = end_time - start_time


# ------------------------------------------------------------
# 8. Make predictions manually
# ------------------------------------------------------------

start_time = time.perf_counter()

y_pred_manual_reg = (
    X_test_reg_bias @ beta_manual
)

end_time = time.perf_counter()

manual_linear_prediction_time = end_time - start_time


# ------------------------------------------------------------
# 9. Calculate MAE manually
# ------------------------------------------------------------

manual_mae = np.mean(
    np.abs(
        y_test_reg_np - y_pred_manual_reg
    )
)


# ------------------------------------------------------------
# 10. Calculate RMSE manually
# ------------------------------------------------------------

manual_rmse = np.sqrt(
    np.mean(
        (
            y_test_reg_np
            - y_pred_manual_reg
        ) ** 2
    )
)


# ------------------------------------------------------------
# 11. Calculate R2 manually
# ------------------------------------------------------------

ss_res = np.sum(
    (
        y_test_reg_np
        - y_pred_manual_reg
    ) ** 2
)

ss_tot = np.sum(
    (
        y_test_reg_np
        - np.mean(y_test_reg_np)
    ) ** 2
)

manual_r2 = 1 - (ss_res / ss_tot)


# ------------------------------------------------------------
# 12. Display Results
# ------------------------------------------------------------

print("\nRESULTS")
print("-" * 40)

print(f"MAE             : {manual_mae:.4f}")
print(f"RMSE            : {manual_rmse:.4f}")
print(f"R2 Score        : {manual_r2:.4f}")

print(
    f"Training Time   : "
    f"{manual_linear_training_time:.6f} seconds"
)

print(
    f"Prediction Time : "
    f"{manual_linear_prediction_time:.6f} seconds"
)


# ------------------------------------------------------------
# 13. Sanity Check
# ------------------------------------------------------------

print("\nSANITY CHECK")
print("-" * 40)

print(
    "Training samples:",
    X_train_reg_bias.shape[0]
)

print(
    "Testing samples :",
    X_test_reg_bias.shape[0]
)

print(
    "Predictions     :",
    len(y_pred_manual_reg)
)

FROM-SCRATCH LINEAR REGRESSION

RESULTS
----------------------------------------
MAE             : 0.1088
RMSE            : 0.1481
R2 Score        : 0.1736
Training Time   : 0.015855 seconds
Prediction Time : 0.000085 seconds

SANITY CHECK
----------------------------------------
Training samples: 957
Testing samples : 240
Predictions     : 240


### Logistic regression from scratch

In [101]:
# PART B2 - LOGISTIC REGRESSION FROM SCRATCH

print("=" * 55)
print("FROM-SCRATCH LOGISTIC REGRESSION")
print("=" * 55)

# Use the SAME classification split from Part A

X_train_clf_manual = X_train_clf.copy()
X_test_clf_manual = X_test_clf.copy()

y_train_clf_manual = y_train_clf.copy()
y_test_clf_manual = y_test_clf.copy()


# Handle missing values manually

# Calculate WIP median from TRAINING data only
wip_median_clf = X_train_clf_manual['wip'].median()

X_train_clf_manual['wip'] = (
    X_train_clf_manual['wip'].fillna(wip_median_clf)
)

X_test_clf_manual['wip'] = (
    X_test_clf_manual['wip'].fillna(wip_median_clf)
)


# One-hot encode categorical features

categorical_features_clf = [
    'quarter',
    'department',
    'day'
]

X_train_clf_manual = pd.get_dummies(
    X_train_clf_manual,
    columns=categorical_features_clf,
    dtype=float
)

X_test_clf_manual = pd.get_dummies(
    X_test_clf_manual,
    columns=categorical_features_clf,
    dtype=float
)

# Make test columns identical to training columns
X_test_clf_manual = X_test_clf_manual.reindex(
    columns=X_train_clf_manual.columns,
    fill_value=0
)


# Scale numerical features manually

numeric_features_clf = [
    'team',
    'targeted_productivity',
    'smv',
    'wip',
    'over_time',
    'incentive',
    'idle_time',
    'idle_men',
    'no_of_style_change',
    'no_of_workers'
]

clf_means = X_train_clf_manual[
    numeric_features_clf
].mean()

clf_stds = X_train_clf_manual[
    numeric_features_clf
].std(ddof=0)

clf_stds = clf_stds.replace(0, 1)


X_train_clf_manual[numeric_features_clf] = (
    X_train_clf_manual[numeric_features_clf]
    - clf_means
) / clf_stds


X_test_clf_manual[numeric_features_clf] = (
    X_test_clf_manual[numeric_features_clf]
    - clf_means
) / clf_stds


# Convert to NumPy arrays

X_train_clf_np = (
    X_train_clf_manual.to_numpy(dtype=float)
)

X_test_clf_np = (
    X_test_clf_manual.to_numpy(dtype=float)
)

y_train_clf_np = (
    y_train_clf_manual.to_numpy(dtype=float)
)

y_test_clf_np = (
    y_test_clf_manual.to_numpy(dtype=float)
)


# Add intercept/bias

X_train_clf_bias = np.c_[
    np.ones(X_train_clf_np.shape[0]),
    X_train_clf_np
]

X_test_clf_bias = np.c_[
    np.ones(X_test_clf_np.shape[0]),
    X_test_clf_np
]


# Sigmoid Function

def sigmoid_manual(z):

    # Avoid overflow in exponential calculation
    z = np.clip(z, -500, 500)

    return 1 / (
        1 + np.exp(-z)
    )


# Initialize Logistic Regression parameters

learning_rate = 0.01
iterations = 5000

weights_manual = np.zeros(
    X_train_clf_bias.shape[1]
)


# Train using Gradient Descent

start_time = time.perf_counter()

for i in range(iterations):

    # Linear combination
    z = (
        X_train_clf_bias
        @ weights_manual
    )

    # Convert z into probabilities
    probabilities = sigmoid_manual(z)

    # Calculate error
    error = (
        probabilities
        - y_train_clf_np
    )

    # Calculate gradient
    gradient = (
        X_train_clf_bias.T
        @ error
    ) / len(y_train_clf_np)

    # Update weights
    weights_manual = (
        weights_manual
        - learning_rate * gradient
    )

end_time = time.perf_counter()

manual_logistic_training_time = (
    end_time - start_time
)


# Probability Prediction + Thresholding

start_time = time.perf_counter()

# Calculate test scores
z_test = (
    X_test_clf_bias
    @ weights_manual
)

# Convert scores to probabilities
y_prob_manual_clf = sigmoid_manual(
    z_test
)

# Threshold = 0.5
y_pred_manual_clf = (
    y_prob_manual_clf >= 0.5
).astype(int)

end_time = time.perf_counter()

manual_logistic_prediction_time = (
    end_time - start_time
)


# Calculate TP, TN, FP and FN manually

TP = np.sum(
    (y_test_clf_np == 1)
    &
    (y_pred_manual_clf == 1)
)

TN = np.sum(
    (y_test_clf_np == 0)
    &
    (y_pred_manual_clf == 0)
)

FP = np.sum(
    (y_test_clf_np == 0)
    &
    (y_pred_manual_clf == 1)
)

FN = np.sum(
    (y_test_clf_np == 1)
    &
    (y_pred_manual_clf == 0)
)


# Accuracy

manual_accuracy = (
    (TP + TN)
    /
    (TP + TN + FP + FN)
)


# Precision

if (TP + FP) != 0:

    manual_precision = (
        TP / (TP + FP)
    )

else:

    manual_precision = 0


# Recall

if (TP + FN) != 0:

    manual_recall = (
        TP / (TP + FN)
    )

else:

    manual_recall = 0


# F1 Score

if (
    manual_precision
    + manual_recall
) != 0:

    manual_f1 = (

        2
        * manual_precision
        * manual_recall

        /

        (
            manual_precision
            + manual_recall
        )
    )

else:

    manual_f1 = 0


# Display Results

print("\nRESULTS")
print("-" * 40)

print(f"Accuracy        : {manual_accuracy:.4f}")
print(f"Precision       : {manual_precision:.4f}")
print(f"Recall          : {manual_recall:.4f}")
print(f"F1 Score        : {manual_f1:.4f}")

print(
    f"Training Time   : "
    f"{manual_logistic_training_time:.6f} seconds"
)

print(
    f"Prediction Time : "
    f"{manual_logistic_prediction_time:.6f} seconds"
)


# Sanity Check

print("\nCONFUSION MATRIX VALUES")
print("-" * 40)

print("TP:", TP)
print("TN:", TN)
print("FP:", FP)
print("FN:", FN)

print("\nSANITY CHECK")
print("-" * 40)

print(
    "Training samples:",
    X_train_clf_bias.shape[0]
)

print(
    "Testing samples :",
    X_test_clf_bias.shape[0]
)

print(
    "Predictions     :",
    len(y_pred_manual_clf)
)

print(
    "TP + TN + FP + FN:",
    TP + TN + FP + FN
)

FROM-SCRATCH LOGISTIC REGRESSION

RESULTS
----------------------------------------
Accuracy        : 0.7542
Precision       : 0.7565
Recall          : 0.9831
F1 Score        : 0.8550
Training Time   : 0.140804 seconds
Prediction Time : 0.000368 seconds

CONFUSION MATRIX VALUES
----------------------------------------
TP: 174
TN: 7
FP: 56
FN: 3

SANITY CHECK
----------------------------------------
Training samples: 957
Testing samples : 240
Predictions     : 240
TP + TN + FP + FN: 240


## Part C - Comparison and Optimization

### Comparison

In [98]:
# MODEL COMPARISON

print("PART C - SCIKIT-LEARN VS FROM-SCRATCH COMPARISON")


# LINEAR REGRESSION COMPARISON

linear_comparison = pd.DataFrame({

    'Metric': [
        'MAE',
        'RMSE',
        'R2 Score',
        'Training Time (sec)',
        'Prediction Time (sec)'
    ],

    'Scikit-learn': [
        mae,
        rmse,
        r2,
        linear_training_time,
        linear_prediction_time
    ],

    'From Scratch': [
        manual_mae,
        manual_rmse,
        manual_r2,
        manual_linear_training_time,
        manual_linear_prediction_time
    ]
})


print("\nLINEAR REGRESSION COMPARISON")
print("-" * 65)

print(
    linear_comparison.to_string(index=False)
)


# LOGISTIC REGRESSION COMPARISON

logistic_comparison = pd.DataFrame({

    'Metric': [
        'Accuracy',
        'Precision',
        'Recall',
        'F1 Score',
        'Training Time (sec)',
        'Prediction Time (sec)'
    ],

    'Scikit-learn': [
        accuracy,
        precision,
        recall,
        f1,
        logistic_training_time,
        logistic_prediction_time
    ],

    'From Scratch': [
        manual_accuracy,
        manual_precision,
        manual_recall,
        manual_f1,
        manual_logistic_training_time,
        manual_logistic_prediction_time
    ]
})


print("\nLOGISTIC REGRESSION COMPARISON")
print("-" * 65)

print(
    logistic_comparison.to_string(index=False)
)

PART C - SCIKIT-LEARN VS FROM-SCRATCH COMPARISON

LINEAR REGRESSION COMPARISON
-----------------------------------------------------------------
               Metric  Scikit-learn  From Scratch
                  MAE      0.108771      0.108771
                 RMSE      0.148134      0.148134
             R2 Score      0.173576      0.173576
  Training Time (sec)      0.169736      0.015855
Prediction Time (sec)      0.009558      0.000085

LOGISTIC REGRESSION COMPARISON
-----------------------------------------------------------------
               Metric  Scikit-learn  From Scratch
             Accuracy      0.750000      0.754167
            Precision      0.769585      0.756522
               Recall      0.943503      0.983051
             F1 Score      0.847716      0.855037
  Training Time (sec)      0.139415      0.383005
Prediction Time (sec)      0.010438      0.000361


### Optimizing logistic regression

In [100]:
# OPTIMIZATION OF MANUAL LOGISTIC REGRESSION

print("MANUAL LOGISTIC REGRESSION OPTIMIZATION")


# Binary Cross-Entropy Loss

def binary_cross_entropy(y, p):

    # Avoid log(0)
    p = np.clip(
        p,
        1e-15,
        1 - 1e-15
    )

    return -np.mean(
        y * np.log(p)
        +
        (1 - y) * np.log(1 - p)
    )


# Hyperparameters to test

learning_rates = [
    0.001,
    0.005,
    0.01,
    0.02
]

max_iterations = 10000

tolerance = 1e-7


tuning_results = []

candidate_weights = {}


# Test each learning rate

for lr in learning_rates:

    # Start every model with zero weights
    weights_temp = np.zeros(
        X_train_clf_bias.shape[1]
    )

    previous_loss = float('inf')

    start_time = time.perf_counter()


    # Gradient Descent
    for iteration in range(max_iterations):

        z = (
            X_train_clf_bias
            @ weights_temp
        )

        probabilities = sigmoid_manual(z)

        error = (
            probabilities
            - y_train_clf_np
        )

        gradient = (
            X_train_clf_bias.T
            @ error
        ) / len(y_train_clf_np)

        weights_temp = (
            weights_temp
            - lr * gradient
        )



        updated_probabilities = sigmoid_manual(
            X_train_clf_bias
            @ weights_temp
        )

        current_loss = binary_cross_entropy(
            y_train_clf_np,
            updated_probabilities
        )



        if abs(
            previous_loss - current_loss
        ) < tolerance:

            break

        previous_loss = current_loss


    training_time_temp = (
        time.perf_counter()
        - start_time
    )


    # Make Test Predictions

    prediction_start = time.perf_counter()

    test_probabilities = sigmoid_manual(
        X_test_clf_bias
        @ weights_temp
    )

    predictions_temp = (
        test_probabilities >= 0.5
    ).astype(int)

    prediction_time_temp = (
        time.perf_counter()
        - prediction_start
    )


    # Manual Confusion Matrix

    TP_temp = np.sum(
        (y_test_clf_np == 1)
        &
        (predictions_temp == 1)
    )

    TN_temp = np.sum(
        (y_test_clf_np == 0)
        &
        (predictions_temp == 0)
    )

    FP_temp = np.sum(
        (y_test_clf_np == 0)
        &
        (predictions_temp == 1)
    )

    FN_temp = np.sum(
        (y_test_clf_np == 1)
        &
        (predictions_temp == 0)
    )


    # Calculate Metrics

    accuracy_temp = (
        (TP_temp + TN_temp)
        /
        len(y_test_clf_np)
    )


    if (TP_temp + FP_temp) != 0:

        precision_temp = (
            TP_temp
            /
            (TP_temp + FP_temp)
        )

    else:

        precision_temp = 0


    if (TP_temp + FN_temp) != 0:

        recall_temp = (
            TP_temp
            /
            (TP_temp + FN_temp)
        )

    else:

        recall_temp = 0


    if (
        precision_temp
        + recall_temp
    ) != 0:

        f1_temp = (
            2
            * precision_temp
            * recall_temp
            /
            (
                precision_temp
                + recall_temp
            )
        )

    else:

        f1_temp = 0


    # Save Results

    tuning_results.append([
        lr,
        iteration + 1,
        current_loss,
        accuracy_temp,
        precision_temp,
        recall_temp,
        f1_temp,
        training_time_temp,
        prediction_time_temp
    ])

    candidate_weights[lr] = (
        weights_temp.copy()
    )


# Create Tuning Results Table

tuning_df = pd.DataFrame(

    tuning_results,

    columns=[
        'Learning Rate',
        'Iterations',
        'Training Loss',
        'Accuracy',
        'Precision',
        'Recall',
        'F1 Score',
        'Training Time',
        'Prediction Time'
    ]
)


print("\nLEARNING RATE TUNING RESULTS")
print("-" * 100)

print(
    tuning_df.to_string(index=False)
)


# Select Final Model

best_index = (
    tuning_df[
        'Training Loss'
    ].idxmin()
)

best_learning_rate = (
    tuning_df.loc[
        best_index,
        'Learning Rate'
    ]
)

best_iterations = int(
    tuning_df.loc[
        best_index,
        'Iterations'
    ]
)

best_loss = (
    tuning_df.loc[
        best_index,
        'Training Loss'
    ]
)

accuracy_opt = (
    tuning_df.loc[
        best_index,
        'Accuracy'
    ]
)

precision_opt = (
    tuning_df.loc[
        best_index,
        'Precision'
    ]
)

recall_opt = (
    tuning_df.loc[
        best_index,
        'Recall'
    ]
)

f1_opt = (
    tuning_df.loc[
        best_index,
        'F1 Score'
    ]
)

optimized_training_time = (
    tuning_df.loc[
        best_index,
        'Training Time'
    ]
)

optimized_prediction_time = (
    tuning_df.loc[
        best_index,
        'Prediction Time'
    ]
)


# Display Selected Model

print(
    "\nSELECTED OPTIMIZED MODEL"
)

print("-" * 50)

print(
    f"Learning Rate   : "
    f"{best_learning_rate}"
)

print(
    f"Iterations Used : "
    f"{best_iterations}"
)

print(
    f"Training Loss   : "
    f"{best_loss:.6f}"
)

print(
    f"Accuracy        : "
    f"{accuracy_opt:.4f}"
)

print(
    f"Precision       : "
    f"{precision_opt:.4f}"
)

print(
    f"Recall          : "
    f"{recall_opt:.4f}"
)

print(
    f"F1 Score        : "
    f"{f1_opt:.4f}"
)

print(
    f"Training Time   : "
    f"{optimized_training_time:.6f} seconds"
)

print(
    f"Prediction Time : "
    f"{optimized_prediction_time:.6f} seconds"
)


# Final Comparison

final_comparison = pd.DataFrame({

    'Metric': [
        'Accuracy',
        'Precision',
        'Recall',
        'F1 Score',
        'Training Time',
        'Prediction Time'
    ],

    'Scikit-learn': [
        accuracy,
        precision,
        recall,
        f1,
        logistic_training_time,
        logistic_prediction_time
    ],

    'Manual Original': [
        manual_accuracy,
        manual_precision,
        manual_recall,
        manual_f1,
        manual_logistic_training_time,
        manual_logistic_prediction_time
    ],

    'Manual Optimized': [
        accuracy_opt,
        precision_opt,
        recall_opt,
        f1_opt,
        optimized_training_time,
        optimized_prediction_time
    ]
})


print(
    "\nFINAL LOGISTIC REGRESSION COMPARISON"
)

print("-" * 85)

print(
    final_comparison.to_string(
        index=False
    )
)

MANUAL LOGISTIC REGRESSION OPTIMIZATION

LEARNING RATE TUNING RESULTS
----------------------------------------------------------------------------------------------------
 Learning Rate  Iterations  Training Loss  Accuracy  Precision   Recall  F1 Score  Training Time  Prediction Time
         0.001       10000       0.524333  0.745833   0.743697 1.000000  0.853012       0.707691         0.000038
         0.005       10000       0.502449  0.754167   0.756522 0.983051  0.855037       0.673301         0.000024
         0.010       10000       0.495133  0.754167   0.765766 0.960452  0.852130       0.766256         0.000026
         0.020       10000       0.491912  0.762500   0.777778 0.949153  0.854962       0.872650         0.000024

SELECTED OPTIMIZED MODEL
--------------------------------------------------
Learning Rate   : 0.02
Iterations Used : 10000
Training Loss   : 0.491912
Accuracy        : 0.7625
Precision       : 0.7778
Recall          : 0.9492
F1 Score        : 0.8550
Training